In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\navon\OneDrive\Desktop\CrossRank")
panel = pd.read_csv(ROOT / "data/processed/uk_etf_eligible_panel.csv", parse_dates=["date"])

eligible = panel[panel["eligible"]].copy()

# cross-sectional benchmark: equal-weighted mean return of the eligible
# universe, computed separately for EACH month — this is the "peer average"
cross_sectional_mean = eligible.groupby("date")["ret"].transform("mean")
eligible["xs_excess_ret"] = eligible["ret"] - cross_sectional_mean

# sanity checks
print(f"Eligible rows with target: {eligible['xs_excess_ret'].notna().sum():,}")
print(eligible["xs_excess_ret"].describe())

# the mean excess return within each month should be ~0 by construction — verify
monthly_check = eligible.groupby("date")["xs_excess_ret"].mean()
print(f"\nMax |monthly mean excess return| (should be ~0): {monthly_check.abs().max():.2e}")

Eligible rows with target: 40,144
count    4.014400e+04
mean    -3.539970e-19
std      3.480151e-02
min     -3.550351e-01
25%     -1.688552e-02
50%     -4.817338e-04
75%      1.624125e-02
max      6.783063e-01
Name: xs_excess_ret, dtype: float64

Max |monthly mean excess return| (should be ~0): 1.65e-17


In [2]:
out_path = ROOT / "data/processed/uk_etf_targets.csv"
eligible[["ticker","date","ret","xs_excess_ret"]].to_csv(out_path, index=False)

import os
print(f"Saved: {out_path}")
print(f"Size: {round(os.path.getsize(out_path)/1e6, 2)} MB")
print(f"Rows: {len(eligible):,}")

Saved: C:\Users\navon\OneDrive\Desktop\CrossRank\data\processed\uk_etf_targets.csv
Size: 2.31 MB
Rows: 40,144


In [4]:
prices = pd.read_csv(ROOT / "data/processed/uk_etf_prices.csv", parse_dates=["date"])
print(f"Prices loaded: {len(prices):,} rows, {prices['ticker'].nunique()} tickers")
prices.head()

Prices loaded: 2,707,129 rows, 1534 tickers


,ticker,date,open,high,low,close,volume
0,100D,2017-11-09,113.859014,113.859014,113.859014,113.859014,0.0
1,100D,2017-11-10,113.080713,113.080713,113.080713,113.080713,0.0
2,100D,2017-11-13,112.808896,112.808896,112.808896,112.808896,0.0
3,100D,2017-11-14,112.797041,112.797041,112.797041,112.797041,0.0
4,100D,2017-11-15,112.160488,112.160488,112.160488,112.160488,0.0


In [5]:
prices_sorted = prices.sort_values(["ticker", "date"]).copy()
prices_sorted["daily_ret"] = prices_sorted.groupby("ticker")["close"].pct_change()

def pct_return(n_days):
    return prices_sorted.groupby("ticker")["close"].transform(
        lambda s: s / s.shift(n_days) - 1
    )

# momentum features — trailing N-day return as of each date
prices_sorted["mom_1m"]  = pct_return(21)
prices_sorted["mom_3m"]  = pct_return(63)
prices_sorted["mom_6m"]  = pct_return(126)
prices_sorted["mom_12m"] = pct_return(252)

# realized volatility — trailing 60-day daily return std, annualized
prices_sorted["realized_vol"] = (
    prices_sorted.groupby("ticker")["daily_ret"]
    .transform(lambda s: s.rolling(60, min_periods=40).std() * (252 ** 0.5))
)

# max drawdown over trailing 252 trading days (~12 months)
def rolling_max_drawdown(s, window=252):
    roll_max = s.rolling(window, min_periods=60).max()
    dd = s / roll_max - 1
    return dd.rolling(window, min_periods=60).min()

prices_sorted["max_drawdown_12m"] = (
    prices_sorted.groupby("ticker")["close"].transform(rolling_max_drawdown)
)

print(f"Rows: {len(prices_sorted):,}")
print(prices_sorted[["mom_1m","mom_3m","mom_6m","mom_12m","realized_vol","max_drawdown_12m"]].describe())

Rows: 2,707,129
             mom_1m        mom_3m        mom_6m       mom_12m  realized_vol  \
count  2.673912e+06  2.610776e+06  2.520245e+06  2.351826e+06  2.646382e+06   
mean   2.035675e-01  2.497974e-01  2.838904e-01  3.060572e-01  3.094291e+00   
std    4.423125e+00  4.789848e+00  4.978614e+00  4.840696e+00  2.810277e+01   
min   -9.989837e-01 -9.989716e-01 -9.988999e-01 -9.987728e-01  0.000000e+00   
25%   -1.863049e-02 -1.999728e-02 -1.941329e-02 -9.917637e-03  1.214380e-01   
50%    1.000310e-02  2.759095e-02  5.060093e-02  9.202123e-02  1.566311e-01   
75%    3.674691e-02  7.166150e-02  1.162646e-01  1.956800e-01  2.150033e-01   
max    2.168246e+02  2.394779e+02  2.623436e+02  2.729145e+02  5.679221e+02   

       max_drawdown_12m  
count      2.532694e+06  
mean      -2.207707e-01  
std        1.565791e-01  
min       -9.999445e-01  
25%       -2.736964e-01  
50%       -1.829999e-01  
75%       -1.201500e-01  
max        0.000000e+00  


In [7]:
features = prices_sorted[["ticker","date","mom_1m","mom_3m","mom_6m","mom_12m",
                            "realized_vol","max_drawdown_12m"]].copy()

features_monthly = (
    features.set_index("date")
    .groupby("ticker")
    .resample("ME")
    .last()
)

# ticker is now a MultiIndex level, not a column — reset_index() recovers both
features_monthly = features_monthly.reset_index()
if "ticker" not in features_monthly.columns:
    # some pandas versions name the outer level differently — recover it explicitly
    features_monthly = features_monthly.rename(columns={features_monthly.columns[0]: "ticker"})

print(features_monthly.columns.tolist())
features_monthly.head()

['ticker', 'date', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'realized_vol', 'max_drawdown_12m']


,ticker,date,mom_1m,mom_3m,mom_6m,mom_12m,realized_vol,max_drawdown_12m
0,100D,2017-11-30,NaN,NaN,NaN,NaN,NaN,NaN
1,100D,2017-12-31,0.015974,NaN,NaN,NaN,NaN,NaN
2,100D,2018-01-31,-0.014591,NaN,NaN,NaN,0.089193,NaN
3,100D,2018-02-28,-0.040972,-0.037930,NaN,NaN,0.121784,NaN
4,100D,2018-03-31,-0.020375,-0.072351,NaN,NaN,0.127740,NaN


In [8]:
eligible_features = eligible.merge(features_monthly, on=["ticker","date"], how="left")

print(f"Eligible rows: {len(eligible_features):,}")
print(eligible_features[["mom_1m","mom_3m","mom_6m","mom_12m","realized_vol","max_drawdown_12m"]].describe())

Eligible rows: 40,144
             mom_1m        mom_3m        mom_6m       mom_12m  realized_vol  \
count  40144.000000  40144.000000  40144.000000  40144.000000  40144.000000   
mean       0.009399      0.035026      0.071344      0.137591      0.330418   
std        0.051609      0.813126      0.984932      1.296279      7.479340   
min       -0.400000     -0.535993     -0.530408     -0.983689      0.037815   
25%       -0.018691     -0.014513     -0.005796      0.015417      0.119781   
50%        0.010306      0.029414      0.060802      0.113712      0.154285   
75%        0.038074      0.073125      0.125356      0.214363      0.207936   
max        0.757001    117.300929    116.414292    131.161988    418.292495   

       max_drawdown_12m  
count      40144.000000  
mean          -0.193180  
std            0.109629  
min           -0.995269  
25%           -0.251643  
50%           -0.170385  
75%           -0.116011  
max           -0.021907  


In [10]:
meta = pd.read_csv(ROOT / "data/processed/uk_etf_metadata.csv")
print(f"Metadata loaded: {len(meta)} tickers")

Metadata loaded: 1534 tickers


In [11]:
worst = eligible_features.nlargest(15, "mom_12m")[["ticker","date","mom_12m","mom_6m","mom_3m","mom_1m"]]
worst_named = worst.merge(meta[["ticker","name"]], on="ticker", how="left")
print(worst_named)

   ticker       date     mom_12m    mom_6m    mom_3m    mom_1m  \
0    S5SD 2026-08-31  131.161988  0.122336  0.027208  0.048641   
1    S5SD 2026-07-31  128.647133  0.103223  0.049024 -0.010321   
2    S5SD 2026-09-30  127.266783  0.195761  0.038128  0.024631   
3    S5SD 2026-10-31  127.054464  0.176042  0.037693  0.014650   
4    DAGB 2023-12-31    3.221730  0.589266  1.075200  0.745861   
5    HYGG 2026-05-31    3.168160  1.073270  0.880606  0.407660   
6    HYGG 2026-04-30    2.654664  0.430804  0.526198  0.644927   
7    CSKR 2026-05-31    2.527103  1.372590  0.400025  0.331373   
8    KRWL 2026-05-31    2.526785  1.342758  0.267594  0.320960   
9    FLRK 2026-05-31    2.514231  1.348267  0.435412  0.354021   
10   XKS2 2026-05-31    2.494083  1.305849  0.409629  0.326975   
11   IKOR 2026-05-31    2.481596  1.291960  0.399412  0.327197   
12   HKOR 2026-05-31    2.469694  1.296218  0.406439  0.332528   
13   JEDG 2026-05-31    2.324927  1.357668  0.606946  0.389346   
14   BCHS 

In [12]:
# exclude the confirmed bad ticker
eligible_features = eligible_features[eligible_features["ticker"] != "S5SD"].copy()

# winsorize momentum/vol features at 1st/99th percentile — caps extreme
# but real values rather than deleting them
feature_cols = ["mom_1m","mom_3m","mom_6m","mom_12m","realized_vol"]

for col in feature_cols:
    lo, hi = eligible_features[col].quantile([0.01, 0.99])
    eligible_features[col] = eligible_features[col].clip(lo, hi)

print(f"Rows after excluding S5SD: {len(eligible_features):,}")
print(eligible_features[feature_cols + ["max_drawdown_12m"]].describe())

Rows after excluding S5SD: 40,092
             mom_1m        mom_3m        mom_6m       mom_12m  realized_vol  \
count  40092.000000  40092.000000  40092.000000  40092.000000  40092.000000   
mean       0.009211      0.028851      0.061693      0.122054      0.183537   
std        0.047141      0.078783      0.112646      0.171835      0.109320   
min       -0.122709     -0.201288     -0.227076     -0.273417      0.076672   
25%       -0.018695     -0.014531     -0.005850      0.015337      0.119817   
50%        0.010303      0.029407      0.060772      0.113652      0.154331   
75%        0.038074      0.073125      0.125351      0.214325      0.207992   
max        0.148398      0.273551      0.431194      0.723902      0.847935   

       max_drawdown_12m  
count      40092.000000  
mean          -0.193038  
std            0.108864  
min           -0.995269  
25%           -0.251643  
50%           -0.170488  
75%           -0.116060  
max           -0.021907  


In [13]:
out_path = ROOT / "data/processed/uk_etf_features_targets.csv"

final_cols = ["ticker","date","ret","xs_excess_ret",
              "mom_1m","mom_3m","mom_6m","mom_12m",
              "realized_vol","max_drawdown_12m"]

eligible_features[final_cols].to_csv(out_path, index=False)

import os
print(f"Saved: {out_path}")
print(f"Size: {round(os.path.getsize(out_path)/1e6, 2)} MB")
print(f"Rows: {len(eligible_features):,}")
print(f"Date range: {eligible_features['date'].min().date()} to {eligible_features['date'].max().date()}")
print(f"Unique tickers: {eligible_features['ticker'].nunique()}")

Saved: C:\Users\navon\OneDrive\Desktop\CrossRank\data\processed\uk_etf_features_targets.csv
Size: 7.2 MB
Rows: 40,092
Date range: 2010-03-31 to 2026-10-31
Unique tickers: 825
